In [ ]:
import os
from pathlib import Path

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pass_pclr.datasets import infer_dataset_class_from_path

In [ ]:
output_dir = Path("../outputs")
datasets = {
    "EchoNext": {
        72475: "runs-echonext",
        32768: "runs-echonext-32k",
        16384: "runs-echonext-16k",
        8192: "runs-echonext-8k",
        4096: "runs-echonext-4k",
        2048: "runs-echonext-2k",
        1024: "runs-echonext-1k",
    },
    "PTB-XL": {
        17418: "runs-ptbxl",
        8722: "runs-ptbxl-8k",
        4356: "runs-ptbxl-4k",
        2175: "runs-ptbxl-2k",
        1091: "runs-ptbxl-1k",
    },
    "CinC Georgia": {
        8192: "runs-cinc",
        4096: "runs-cinc-4k",
        2048: "runs-cinc-2k",
        1024: "runs-cinc-1k",
    },
    "MIMIC-IV-ECG (ED)": {
        78470: "runs-mimic",
        32768: "runs-mimic-32k",
        16384: "runs-mimic-16k",
        8192: "runs-mimic-8k",
        4096: "runs-mimic-4k",
        2048: "runs-mimic-2k",
        1024: "runs-mimic-1k",
    },
    "ZZU pediatric ECG": {
        8658: "runs-zzu",
        4096: "runs-zzu-4k",
        2048: "runs-zzu-2k",
        1024: "runs-zzu-1k",
    },
}

# mapping of experiment name to tuple of:
# - plotting color
# - folder name
experiments = {
    "Blackbox Direct": ("tab:grey",   "blackbox-direct"),
    "ECGFounder (LR)": ("tab:red",    "ecgfounder-logreg"),
    ###
    "LabSup Proto Direct (14ppl)":      ("tab:green",  "labsup-proto-direct-14ppl"),
    "LabSup Proto Direct (14ppl) (LR)": ("tab:green",  "labsup-proto-direct-14ppl-logreg"),
    ###
    "ProtoSSL HEEDB (PIA) (14ppl)":      ("tab:blue",   "protossl-heedb-pia-14ppl"),
    "ProtoSSL HEEDB (PIA) (14ppl) (LR)": ("tab:blue",   "protossl-heedb-pia-14ppl-logreg"),
    ###
    "ProtoSSL HEEDB (PIA) (14ppl) (no-attn)":      ("tab:purple",   "protossl-heedb-no-attn-pia-14ppl"),
    "ProtoSSL HEEDB (PIA) (14ppl) (no-attn) (LR)": ("tab:purple",   "protossl-heedb-no-attn-pia-14ppl-logreg"),
    ###
    "ProtoSSL HEEDB (PILA) (14ppl) (no-attn)":      ("tab:cyan",   "protossl-heedb-no-attn-pila-14ppl"),
    "ProtoSSL HEEDB (PILA) (14ppl) (no-attn) (LR)": ("tab:cyan",   "protossl-heedb-no-attn-pila-14ppl-logreg"),
    ###
    "ProtoSSL HEEDB (PILMA) (14ppl) (no-attn)":      ("tab:blue",   "protossl-heedb-no-attn-pilma-14ppl"),
    "ProtoSSL HEEDB (PILMA) (14ppl) (no-attn) (LR)": ("tab:blue",   "protossl-heedb-no-attn-pilma-14ppl-logreg"),
    ###
    "LabSup Proto HEEDB (RIA) (14ppl)":      ("tab:orange", "labsup-proto-heedb-ria-14ppl"),
    "LabSup Proto HEEDB (RIA) (14ppl) (LR)": ("tab:orange", "labsup-proto-heedb-ria-14ppl-logreg"),
    ###
    "LabSup Proto HEEDB (RILA) (14ppl)":      ("tab:pink", "labsup-proto-heedb-rila-14ppl"),
    "LabSup Proto HEEDB (RILA) (14ppl) (LR)": ("tab:pink", "labsup-proto-heedb-rila-14ppl-logreg"),
    ###
    "LabSup Proto HEEDB (RILMA) (14ppl)":      ("tab:brown", "labsup-proto-heedb-rilma-14ppl"),
    "LabSup Proto HEEDB (RILMA) (14ppl) (LR)": ("tab:brown", "labsup-proto-heedb-rilma-14ppl-logreg"),
}


def get_palette(exp_names):
    palette = dict()
    exps = experiments
    for exp_name in exp_names:
        palette[exp_name] = exps[exp_name][0]
    return palette

In [ ]:
data = []
for ds, sizes in datasets.items():
    for size, run_dir in sizes.items():
        _, labels = infer_dataset_class_from_path(run_dir)
        exps = experiments.copy()
        for exp_name, (exp_color, exp_dir) in exps.items():
            metrics_csv = output_dir / run_dir / exp_dir / "metrics.csv"
            if not os.path.exists(metrics_csv):
                continue
            metrics = pd.read_csv(metrics_csv, index_col="Label")
            multilabel = metrics.loc["Multilabel Averaged"]
            datum = {
                "Dataset": ds,
                "Model": exp_name,
                "Train Size": size,
                "Multilabel (AUROC)": multilabel["AUROC"],
                "Multilabel (AUPRC)": multilabel["AUPRC"],
            }
            data.append(datum)
results = pd.DataFrame.from_records(data)

In [ ]:
def plot_lift(
    *,  # enforce kwargs
    df: pd.DataFrame, # long format df (each point to plot is a row)
    dataset: str,
    metric: str,
    models: list[str], # must be intentional about which models to plot
    rename: list[str] | None = None,
    baseline_model: str | None = None, # singular result to optionally plot as dashed line
    ylim: tuple[float, float] | None = None,
    xlim: tuple[float, float] | None = None,
    save_path: str | None = None,
):
    if rename is not None:
        assert len(models) == len(rename)
    if baseline_model is not None and baseline_model not in models:
        models = [baseline_model] + models
        if rename is not None:
            rename = [baseline_model] + rename
    palette = get_palette(models)
    if rename is not None:
        palette = {new_name: palette[m] for new_name, m in zip(rename, models)}
        df = df.copy()
        df["Model"] = df["Model"].replace({v: k for k, v in zip(rename, models)})

    df = df[df["Dataset"] == dataset]
    fig, ax = plt.subplots(figsize=(6, 6))
    min_size = df["Train Size"].min()
    max_size = df["Train Size"].max()
    if xlim is not None:
        min_size = min(min_size, xlim[0])
        max_size = max(max_size, xlim[1])

    if baseline_model is not None:
        mask = df["Model"] == baseline_model
        assert (
            mask.sum() == 1
        ), f"Should only have 1 entry for baseline model: {baseline_model}"
        baseline_row = df[mask].iloc[0]
        ax.hlines(
            baseline_row[metric],
            min_size,
            max_size,
            colors=palette.pop(baseline_model),
            linestyles=":",
            label=baseline_model,
        )
        df = df[~mask] # subsequent line plots should exclude baseline model

    sns.lineplot(
        df,
        x="Train Size",
        y=metric,
        hue="Model",
        palette=palette,
        hue_order=list(palette.keys()),
        marker="o",
        ax=ax,
    )
    ax.set_xscale("log", base=2)
    if xlim is not None:
        ax.set_xlim(xlim)
    else:
        # tighter boundaries than default lims
        ax.set_xlim((min_size, max_size))
    ax.set_title(f"{dataset} {metric}")
    ax.legend(loc="lower right")
    if ylim is not None:
        ax.set_ylim(ylim)
    if save_path is not None:
        fig.tight_layout()
        fig.savefig(save_path)

In [ ]:
# Best Results

plot_lift(
    df=results,
    dataset="EchoNext",
    metric="Multilabel (AUROC)",
    models=[
        "Blackbox Direct",
        "ECGFounder (LR)",
        "LabSup Proto Direct (14ppl) (LR)",
        "LabSup Proto HEEDB (RIA) (14ppl) (LR)",
        "LabSup Proto HEEDB (RILA) (14ppl) (LR)",
        "ProtoSSL HEEDB (PIA) (14ppl) (no-attn) (LR)",
        "ProtoSSL HEEDB (PILA) (14ppl) (no-attn) (LR)",
    ],
    rename=[
        "Blackbox Direct",
        "ECGFounder",
        "LabSup Proto Direct",
        "LabSup Proto HEEDB (RIA)",
        "LabSup Proto HEEDB (RILA)",
        "ProtoSSL HEEDB (PIA)",
        "ProtoSSL HEEDB (PILA)",
    ],
    save_path="figs/echonext-multilabel.png",
)

plot_lift(
    df=results,
    dataset="PTB-XL",
    metric="Multilabel (AUROC)",
    models=[
        "Blackbox Direct",
        "ECGFounder (LR)",
        "LabSup Proto Direct (14ppl) (LR)",
        "LabSup Proto HEEDB (RIA) (14ppl) (LR)",
        "LabSup Proto HEEDB (RILA) (14ppl) (LR)",
        "ProtoSSL HEEDB (PIA) (14ppl) (no-attn) (LR)",
        "ProtoSSL HEEDB (PILA) (14ppl) (no-attn) (LR)",
    ],
    rename=[
        "Blackbox Direct",
        "ECGFounder",
        "LabSup Proto Direct",
        "LabSup Proto HEEDB (RIA)",
        "LabSup Proto HEEDB (RILA)",
        "ProtoSSL HEEDB (PIA)",
        "ProtoSSL HEEDB (PILA)",
    ],
    save_path="figs/ptbxl-multilabel.png",
)

plot_lift(
    df=results,
    dataset="CinC Georgia",
    metric="Multilabel (AUROC)",
    models=[
        "Blackbox Direct",
        "ECGFounder (LR)",
        "LabSup Proto Direct (14ppl) (LR)",
        "LabSup Proto HEEDB (RIA) (14ppl) (LR)",
        "LabSup Proto HEEDB (RILA) (14ppl) (LR)",
        "ProtoSSL HEEDB (PIA) (14ppl) (no-attn) (LR)",
        "ProtoSSL HEEDB (PILA) (14ppl) (no-attn) (LR)",
    ],
    rename=[
        "Blackbox Direct",
        "ECGFounder",
        "LabSup Proto Direct",
        "LabSup Proto HEEDB (RIA)",
        "LabSup Proto HEEDB (RILA)",
        "ProtoSSL HEEDB (PIA)",
        "ProtoSSL HEEDB (PILA)",
    ],
    save_path="figs/cinc-multilabel.png",
)

plot_lift(
    df=results,
    dataset="MIMIC-IV-ECG (ED)",
    metric="Multilabel (AUROC)",
    models=[
        "Blackbox Direct",
        "ECGFounder (LR)",
        "LabSup Proto Direct (14ppl) (LR)",
        "LabSup Proto HEEDB (RIA) (14ppl) (LR)",
        "LabSup Proto HEEDB (RILA) (14ppl) (LR)",
        "ProtoSSL HEEDB (PIA) (14ppl) (no-attn) (LR)",
        "ProtoSSL HEEDB (PILA) (14ppl) (no-attn) (LR)",
    ],
    rename=[
        "Blackbox Direct",
        "ECGFounder",
        "LabSup Proto Direct",
        "LabSup Proto HEEDB (RIA)",
        "LabSup Proto HEEDB (RILA)",
        "ProtoSSL HEEDB (PIA)",
        "ProtoSSL HEEDB (PILA)",
    ],
    save_path="figs/mimic-multilabel.png",
)

plot_lift(
    df=results,
    dataset="ZZU pediatric ECG",
    metric="Multilabel (AUROC)",
    models=[
        "Blackbox Direct",
        "ECGFounder (LR)",
        "LabSup Proto Direct (14ppl) (LR)",
        "LabSup Proto HEEDB (RIA) (14ppl) (LR)",
        "LabSup Proto HEEDB (RILA) (14ppl) (LR)",
        "ProtoSSL HEEDB (PIA) (14ppl) (no-attn) (LR)",
        "ProtoSSL HEEDB (PILA) (14ppl) (no-attn) (LR)",
    ],
    rename=[
        "Blackbox Direct",
        "ECGFounder",
        "LabSup Proto Direct",
        "LabSup Proto HEEDB (RIA)",
        "LabSup Proto HEEDB (RILA)",
        "ProtoSSL HEEDB (PIA)",
        "ProtoSSL HEEDB (PILA)",
    ],
    save_path="figs/zzu-multilabel.png",
)